In [2]:
# Import necessary modules
# glob allows unix style pathname pattern expansion
from glob import glob
# pydicom is the python dicom reader
import pydicom as dicom
import nibabel.nicom.csareader as csareader
import os
import pandas as pd

/tmp/ipykernel_3071283/1246310376.py:6: UserWarning: The DICOM readers are highly experimental, unstable, and only work for Siemens time-series at the moment
Please use with caution.  We would be grateful for your help in improving them
  import nibabel.nicom.csareader as csareader


In [3]:
# 5-T1w_MPR_vNav -- MPRAGE structrual dicoms
# 6-fMRI_DistortionMap_PA -- fmri field map dicoms
# 7-fMRI_DistortionMap_AP -- fmri field map dicoms opposite phase encode direction
# 9-fMRI_REVL_ROI_loc_2 -- fmri localizer dicoms run1
# 10-fMRI_REVL_Study_1 -- fmri task dicoms run1
# 14-dMRI_DistortionMap_AP_dMRI_REVL -- dwi field map dicoms
# 16-dMRI_AP_REVL -- diffusion weighted dicoms

print(os.getcwd())
user_ad = 'eskog001'

# directories will have to be specific to your computer
base_dir = f'/home/{user_ad}/Mattfeld_PSB6351_eas'
proj_dir = '/sourcedata'


/home/eskog001/Mattfeld_PSB6351_eas/code/conversion


In [4]:
# Elements we're interested in
# Repetition Time
# Echo Time
# Acquisition Matrix 
# Flip Angle
# Acquisition Number
# Protocol Name
# Slice times = data_set[0x00191029].value

scans_of_interest = ['5-T1w_MPR_vNav',
                    '6-fMRI_DistortionMap_PA',
                    '7-fMRI_DistortionMap_AP',
                    '9-fMRI_REVL_ROI_loc_2',
                    '10-fMRI_REVL_Study_1',
                    '14-dMRI_DistortionMap_AP_dMRI_REVL',
                    '16-dMRI_AP_REVL']

for curr_scan in scans_of_interest:
    
    subj_dir = f'/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/{curr_scan}/resources/DICOM/files'

    # with strings I can concatenate them with simple addition lines
    data_dir = base_dir + proj_dir + subj_dir

    print(data_dir)

/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/5-T1w_MPR_vNav/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/6-fMRI_DistortionMap_PA/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/7-fMRI_DistortionMap_AP/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/9-fMRI_REVL_ROI_loc_2/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/10-fMRI_REVL_Study_1/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/14-dMRI_DistortionMap_AP_dMRI_REVL/resources/DICOM/files
/home/eskog001/Mattfeld_PSB6351_eas/sourcedata/sub-021_extracted/Mattfeld_REVL-000-vCAT-021-S1/scans/16-dMRI_AP_REVL/reso

In [5]:
    # Here I use glob to grab the dicom files
    # Why would I use glob?  What does it give me?

    #glob allows the selection of all files (.txt, .csv, etc) within a dir
    data_files = sorted(glob(data_dir + '/*'))

    print('')
    print('')
    print(data_files[0].split('/')[7])

    mri_vol = dicom.dcmread(data_files[0])

    print(mri_vol)
    
    print('The number of TRs collected equals: {0}'.format(len(data_files)))
    print('SOMETHING HERE RELATED TIMING OF REPETITION TIME: {0} ms'.format(mri_vol.RepetitionTime))
    print(f'The timing of the TE was: {mri_vol.EchoTime} ms')
    print(f'The flip angle in degrees was: {mri_vol.FlipAngle}')
    print(f'The name of the scan was: {mri_vol.ProtocolName}')
    pe_dir = mri_vol.InPlanePhaseEncodingDirection
    print('The phase encode direction was: {0}'.format(pe_dir))
    print('COL = AP or PA; ROW = RL or LR')
    if 0x00191029 in mri_vol:
        print(f'The slice timing was: {mri_vol[0x00191029].value}')
    print(f'The field of view - FOV was: {mri_vol.PercentPhaseFieldOfView}') #unsure if this is the FOV
    print(f'The Field of view (alt) - FOValt was: {mri_vol[int('0051100c', 16)].value}')

        #(0018,1312) In-plane Phase Encoding Direction   CS: 'COL'
        
    # Why use the following format to access information in the header?
    #print(data_set[int('0051100c', 16)].value) # data_set[0x0051100c].value
    # Allows us to access information that is private in the dicom header.
    
    # How do I find phase encoding direction information?
        #header is called "In Plane Phase Encoding Direction" within the dcm file. the .format function
        #automatically extracts data from that column and formats it for saving/exporting/printing



scans
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 202
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.12.2.1107.5.2.43.166003.201906191633084950922282
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.40.0.13.1.1
(0002,0013) Implementation Version Name         SH: 'dcm4che-2.0'
(0002,0016) Source Application Entity Title     AE: 'AN_MEDCOMNT204'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'DIFFUSION', 'NONE', 'ND', 'MOSAIC']
(0008,0012) Instance Creation Date              DA: '20190619'
(0008,0013) Instance Creation Time              TM: '163435.377000'
(0008,00